In [1]:
from datasets import load_dataset
import pandas as pd
import unicodedata
import re

kea_st_dataset = load_dataset("Iscte-Sintra/Korpora_paralela_st")
print(kea_st_dataset)

/home/erik/dev/LLM-CCV-Translation-v2.0/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['original_iso_code', 'original_sentence', 'translation', 'target_iso_code'],
        num_rows: 21771
    })
    validation: Dataset({
        features: ['original_iso_code', 'original_sentence', 'translation', 'target_iso_code'],
        num_rows: 2721
    })
    test: Dataset({
        features: ['original_iso_code', 'original_sentence', 'translation', 'target_iso_code'],
        num_rows: 2722
    })
})


In [3]:
# Convert splits to pandas
train_df = kea_st_dataset["train"].to_pandas()
val_df = kea_st_dataset["validation"].to_pandas()
test_df = kea_st_dataset["test"].to_pandas()

# Extract unique sentences
train_sentences = set(train_df["original_sentence"].dropna())
val_sentences = set(val_df["original_sentence"].dropna())
test_sentences = set(test_df["original_sentence"].dropna())

# Check intersections
train_val_leakage = train_sentences.intersection(val_sentences)
train_test_leakage = train_sentences.intersection(test_sentences)
val_test_leakage = val_sentences.intersection(test_sentences)

print("Train ↔ Validation:", len(train_val_leakage))
print("Train ↔ Test:", len(train_test_leakage))
print("Validation ↔ Test:", len(val_test_leakage))

Train ↔ Validation: 691
Train ↔ Test: 654
Validation ↔ Test: 140


In [4]:
train_val_leakage

{' 1. AGUESSY, Honorat/BALOGUN, Ola/DIAGNE, Pathé/SOW, Alfa Ibrahima – Intruduson a kultura Afrikána, Union Jeral di Idison, Paris, 1977.',
 ' 1. PARSONS, Elsie Clews – Folclore di ilha di Kábu Berdi, párti II, publikádu pa American Folk-Lore Society, Nóba York, 1923.',
 ' Publikádu: 2024,  Titulu: Tenba un bes dos armun:',
 '(III)», Na Nason, nu 9 (01- 07.11.2007), p.32.',
 '(IV)», na Nason, nu 11 (15- 21.11.2007), p.8.',
 "(Na bida prátiku, sta kláru!) Más, k'el sujetu la ka ê un kalker.",
 '(…) Nunka bu ka ta txiga na ben, ta ánda na kaminhu mariádu.» (p.149).',
 '(…) Ta konta mintira, ta ritrasa, ta frá, ta kalunia, ta afistula, ta ntola di láma, ta funda, ta xuxánu; ta mánxa tudu, tudu ta dismorona; y na fin di kanpánha, pasándu ileison, ta ragánha, ti Kondilus, bóka grándi inponenti, y grasnan. ',
 '* Pamódi ki Bránka di Nevi, txomába Bránka di Névi?',
 '- Bon dia, Nhu da-m benson - Djokinha rusponde-l.',
 '- Bu sabeba ma már pega lumi?',
 '- Dexa di kran-kran, tarse-m kalker un.

In [5]:
import unicodedata
import re

def normalize_sentence(sentence):
    if not isinstance(sentence, str):
        return ""

    # Normalize Unicode
    sentence = unicodedata.normalize("NFKC", sentence)
    # Lowercase
    sentence = sentence.casefold()
    # Remove leading/trailing spaces
    sentence = sentence.strip()
    # Normalize multiple spaces
    sentence = re.sub(r"\s+", " ", sentence)

    return sentence

In [6]:
train_df["normalized_sentence"] = train_df["original_sentence"].apply(normalize_sentence)
val_df["normalized_sentence"] = val_df["original_sentence"].apply(normalize_sentence)
test_df["normalized_sentence"] = test_df["original_sentence"].apply(normalize_sentence)

In [7]:
train_set = set(train_df["normalized_sentence"]) - {""}
val_set = set(val_df["normalized_sentence"]) - {""}
test_set = set(test_df["normalized_sentence"]) - {""}

print("Train ↔ Validation:", len(train_set & val_set))
print("Train ↔ Test:", len(train_set & test_set))
print("Validation ↔ Test:", len(val_set & test_set))

Train ↔ Validation: 692
Train ↔ Test: 660
Validation ↔ Test: 141


In [9]:
def calculate_leakage(reference_df, evaluation_df):
    reference_sentences = set(reference_df["normalized_sentence"]) - {""}
    leaked_rows = evaluation_df[evaluation_df["normalized_sentence"].isin(reference_sentences)]

    leakage_percentage = (len(leaked_rows) / len(evaluation_df) * 100
        if len(evaluation_df) > 0 else 0
    )

    return {
        "total_samples": len(evaluation_df),
        "leaked_samples": len(leaked_rows),
        "leakage_percentage": round(leakage_percentage, 2)
    }


print("Train → Validation:", calculate_leakage(train_df, val_df))
print("Train → Test:", calculate_leakage(train_df, test_df))

Train → Validation: {'total_samples': 2721, 'leaked_samples': 834, 'leakage_percentage': 30.65}
Train → Test: {'total_samples': 2722, 'leaked_samples': 804, 'leakage_percentage': 29.54}


### 1. Identificar todas as frases existentes no conjunto oficial de teste que estejam tambem no conjunto de treino e validação

In [12]:

def normalize_sentence(sentence):
    if not isinstance(sentence, str):
        return None

    sentence = unicodedata.normalize("NFKC", sentence)
    sentence = sentence.casefold()
    sentence = re.sub(r"\s+", " ", sentence).strip()

    return sentence if sentence else None


# Original datasets
train_df = kea_st_dataset["train"].to_pandas()
val_df = kea_st_dataset["validation"].to_pandas()
test_df = kea_st_dataset["test"].to_pandas()

# Normalize sentences
for df in [train_df, val_df, test_df]:
    df["normalized_sentence"] = df["original_sentence"].apply(normalize_sentence)

# Official test sentences
test_sentences = set(test_df["normalized_sentence"].dropna())

# Identify contaminated samples
train_leakage = train_df[train_df["normalized_sentence"].isin(test_sentences)]
val_leakage = val_df[val_df["normalized_sentence"].isin(test_sentences)]

print(f"Train contaminated samples: {len(train_leakage)}")
print(f"Validation contaminated samples: {len(val_leakage)}")

Train contaminated samples: 1990
Validation contaminated samples: 251


### 2. Remover a contaminação do conjunto de teste e de treino

In [13]:
train_clean = train_df[~train_df["normalized_sentence"].isin(test_sentences)].copy()
val_clean = val_df[~val_df["normalized_sentence"].isin(test_sentences)].copy()

### 3. Verificar a contaminação entre o conjunto de treino e validação.

In [14]:
train_sentences = set(train_clean["normalized_sentence"].dropna())
val_sentences = set(val_clean["normalized_sentence"].dropna())

print("Train ↔ Validation:", len(train_sentences & val_sentences))

Train ↔ Validation: 614


### 1. Preparar os dados

In [16]:
import pandas as pd
import unicodedata
import re

from datasets import load_dataset

dataset = load_dataset("Iscte-Sintra/Korpora_paralela_st")

def normalize_sentence(sentence):
    if not isinstance(sentence, str):
        return None

    sentence = unicodedata.normalize("NFKC", sentence)
    sentence = sentence.casefold()
    sentence = re.sub(r"\s+", " ", sentence).strip()

    return sentence or None


# Prepare datasets
dataframes = {}

for split, ds in dataset.items():
    df = ds.to_pandas()
    df["kea_norm"] = df["original_sentence"].apply(normalize_sentence)
    df["pt_norm"] = df["translation"].apply(normalize_sentence)

    dataframes[split] = df
    print(f"{split}: {len(df)} samples")

train: 21771 samples
validation: 2721 samples
test: 2722 samples


In [17]:
def audit_leakage(reference_df, evaluation_df):

    # 1. KEA sentences
    reference_kea = set(reference_df["kea_norm"].dropna())

    # 2. PT sentences
    reference_pt = set(reference_df["pt_norm"].dropna())

    # 3. Translation pairs
    reference_pairs = set(zip(reference_df["kea_norm"], reference_df["pt_norm"]))

    # Remove invalid pairs
    reference_pairs = {
        (kea, pt)
        for kea, pt in reference_pairs
        if kea is not None and pt is not None
    }

    # KEA overlap
    kea_overlap = evaluation_df["kea_norm"].isin(reference_kea)
    # PT overlap
    pt_overlap = evaluation_df["pt_norm"].isin(reference_pt)
    # Exact pair overlap
    pair_overlap = pd.Series(
        [
            (kea, pt) in reference_pairs
            if kea is not None and pt is not None
            else False
            for kea, pt in zip(
                evaluation_df["kea_norm"],
                evaluation_df["pt_norm"]
            )
        ],
        index=evaluation_df.index
    )

    # Any overlap
    any_overlap = kea_overlap | pt_overlap

    # Results
    total = len(evaluation_df)

    results = {
        "total_samples": total,
        "kea_overlap": int(kea_overlap.sum()),
        "pt_overlap": int(pt_overlap.sum()),
        "pair_overlap": int(pair_overlap.sum()),
        "any_overlap": int(any_overlap.sum()),
        "kea_overlap_pct": round(kea_overlap.mean() * 100, 2),
        "pt_overlap_pct": round(pt_overlap.mean() * 100, 2),
        "pair_overlap_pct": round(pair_overlap.mean() * 100, 2),
        "any_overlap_pct": round(any_overlap.mean() * 100, 2)
    }

    # Detailed dataset
    details = evaluation_df.copy()

    details["kea_overlap"] = kea_overlap
    details["pt_overlap"] = pt_overlap
    details["pair_overlap"] = pair_overlap

    # Only overlapping rows
    details = details[any_overlap].copy()

    return results, details

In [18]:
comparisons = [
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test")
]

audit_results = []
audit_details = {}

for reference_split, evaluation_split in comparisons:

    results, details = audit_leakage(
        dataframes[reference_split],
        dataframes[evaluation_split]
    )

    comparison_name = f"{reference_split} -> {evaluation_split}"

    results["comparison"] = comparison_name

    audit_results.append(results)

    audit_details[comparison_name] = details


# Summary
audit_summary = pd.DataFrame(audit_results)

columns = [
    "comparison",
    "total_samples",
    "kea_overlap",
    "kea_overlap_pct",
    "pt_overlap",
    "pt_overlap_pct",
    "pair_overlap",
    "pair_overlap_pct",
    "any_overlap",
    "any_overlap_pct"
]

print(audit_summary[columns].to_string(index=False))

         comparison  total_samples  kea_overlap  kea_overlap_pct  pt_overlap  pt_overlap_pct  pair_overlap  pair_overlap_pct  any_overlap  any_overlap_pct
train -> validation           2721          834            30.65         326           11.98           277             10.18          867            31.86
      train -> test           2722          804            29.54         329           12.09           289             10.62          834            30.64
 validation -> test           2722          244             8.96         165            6.06           118              4.34          272             9.99


In [25]:
train_df = dataframes["train"]
val_df = dataframes["validation"]
test_df = dataframes["test"]


def get_overlap_mask(df, reference_df):
    reference_kea = set(reference_df["kea_norm"].dropna())
    reference_pt = set(reference_df["pt_norm"].dropna())

    kea_overlap = df["kea_norm"].isin(reference_kea)
    pt_overlap = df["pt_norm"].isin(reference_pt)

    return kea_overlap | pt_overlap


# Train examples overlapping with official test
train_test_mask = get_overlap_mask(train_df, test_df)

# Validation examples overlapping with official test
val_test_mask = get_overlap_mask(val_df, test_df)


print("TRAIN")
print(f"Original samples: {len(train_df)}")
print(f"Samples to remove: {train_test_mask.sum()}")
print(f"Percentage: {train_test_mask.mean() * 100:.2f}%")
print(f"Remaining: {(~train_test_mask).sum()}")

print("\nVALIDATION")
print(f"Original samples: {len(val_df)}")
print(f"Samples to remove: {val_test_mask.sum()}")
print(f"Percentage: {val_test_mask.mean() * 100:.2f}%")
print(f"Remaining: {(~val_test_mask).sum()}")

TRAIN
Original samples: 21771
Samples to remove: 2201
Percentage: 10.11%
Remaining: 19570

VALIDATION
Original samples: 2721
Samples to remove: 270
Percentage: 9.92%
Remaining: 2451


In [34]:
results, details = audit_leakage( train_clean, val_clean)
print(pd.Series(results))

# Quantos exemplos do treino removeríamos
# para preservar a validação atual?
train_val_mask = get_overlap_mask(train_clean, val_clean)

print("Additional train samples to remove:", train_val_mask.sum())
print(f"Percentage of remaining train:{train_val_mask.mean() * 100:.2f}%")

total_samples       2451.00
kea_overlap          635.00
pt_overlap           145.00
pair_overlap         112.00
any_overlap          659.00
kea_overlap_pct       25.91
pt_overlap_pct         5.92
pair_overlap_pct       4.57
any_overlap_pct       26.89
dtype: float64
Additional train samples to remove: 883
Percentage of remaining train:4.51%


### Last Step solve the contamination by cleaning the data

In [35]:
# 1. Remove train examples overlapping with official test
train_clean = train_df[~get_overlap_mask(train_df, test_df)].copy()
# 2. Remove validation examples overlapping with official test
val_clean = val_df[~get_overlap_mask(val_df, test_df)].copy()
# 3. Remove train examples overlapping with validation
train_final = train_clean[~get_overlap_mask(train_clean, val_clean)].copy()

# 4. Preserve validation and official test
val_final = val_clean.copy()
test_final = test_df.copy()

print("Final dataset sizes:")
print(f"Train: {len(train_final)}")
print(f"Validation: {len(val_final)}")
print(f"Test: {len(test_final)}")

Final dataset sizes:
Train: 18687
Validation: 2451
Test: 2722


### Check again

In [37]:
final_dataframes = {
    "train": train_final,
    "validation": val_final,
    "test": test_final
}

comparisons = [
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test")
]

for reference, evaluation in comparisons:

    results, _ = audit_leakage(final_dataframes[reference], final_dataframes[evaluation])

    print(f"\n{reference} -> {evaluation}")
    print(f"KEA overlap: {results['kea_overlap']}")
    print(f"PT overlap: {results['pt_overlap']}")
    print(f"Pair overlap: {results['pair_overlap']}")
    print(f"Any overlap: {results['any_overlap']}")

    assert results["any_overlap"] == 0

print("\nNo normalized sentence overlap detected!")


train -> validation
KEA overlap: 0
PT overlap: 0
Pair overlap: 0
Any overlap: 0

train -> test
KEA overlap: 0
PT overlap: 0
Pair overlap: 0
Any overlap: 0

validation -> test
KEA overlap: 0
PT overlap: 0
Pair overlap: 0
Any overlap: 0

No normalized sentence overlap detected!
